# Bronze to Silver

Creates the `adventurework.silver` schema and copies every table from `adventurework.bronze` into it, renaming every table to its data-model name (`product` -> `dim_product`, `purchaseorderdetail` -> `fact_purchases`, `salesterritory` -> `dim_sales_territory`, `salesorderdetail` -> `fact_sales`, `purchaseorderheader` -> `stg_purchaseorderheader`, `vendor` -> `dim_vendor`).

**Prerequisite:** run the *Raw_to_Bronze* notebook first so the bronze tables exist.

**Run order: cell 1 -> cell 2 -> cell 3**

In [0]:
# Create the silver schema
spark.sql("CREATE SCHEMA IF NOT EXISTS adventurework.silver")
print("Schema adventurework.silver is ready")

In [0]:
# Copy every table from adventurework.bronze into adventurework.silver,
# renaming tables to their data-model names where defined
TABLE_RENAMES = {
    "product":             "dim_product",
    "purchaseorderdetail": "fact_purchases",
    "salesterritory":      "dim_sales_territory",
    "salesorderdetail":    "fact_sales",
    "purchaseorderheader": "stg_purchaseorderheader",
    "vendor":              "dim_vendor",
}

bronze_tables = [t.name for t in spark.catalog.listTables("adventurework.bronze")]

if not bronze_tables:
    print("No tables found in adventurework.bronze - run the Raw_to_Bronze notebook first")
else:
    for table_name in sorted(bronze_tables):
        silver_name = TABLE_RENAMES.get(table_name, table_name)
        spark.sql(
            f"CREATE OR REPLACE TABLE adventurework.silver.{silver_name} "
            f"AS SELECT * FROM adventurework.bronze.{table_name}"
        )
        print(f"adventurework.bronze.{table_name} -> adventurework.silver.{silver_name}: copied")

In [0]:
# Verify: row counts in silver must match bronze (accounting for renamed tables)
for t in sorted(spark.catalog.listTables("adventurework.bronze"), key=lambda t: t.name):
    bronze_count = spark.table(f"adventurework.bronze.{t.name}").count()
    silver_name = TABLE_RENAMES.get(t.name, t.name)
    silver_count = spark.table(f"adventurework.silver.{silver_name}").count()
    status = "OK" if silver_count == bronze_count else "MISMATCH"
    print(f"adventurework.bronze.{t.name} -> adventurework.silver.{silver_name}: {silver_count} rows (bronze: {bronze_count}) {status}")